<a href="https://colab.research.google.com/github/Bienbaz/Feature-Engineering-NBA-Player-Longevity-Prediction/blob/main/nba_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Player Longevity Prediction – Feature Engineering Evaluation
This notebook performs feature engineering for predicting `target_5yrs`.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [2]:
# Update path if needed
df = pd.read_csv('nba-players.csv')
df.head()

,Unnamed: 0,name,gp,min,pts,fgm,fga,fg,3p_made,3pa,...,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,0,Brandon Ingram,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,...,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,1,Andrew Harrison,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,...,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,2,JaKarr Sampson,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,...,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,3,Malik Sealy,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,...,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,4,Matt Geiger,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,...,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


## 3. Define Target Variable

In [3]:
target = 'target_5yrs'
y = df[target]
X = df.drop(columns=[target])

## 4. Remove Non-Predictive Columns

In [4]:
noise_cols = [c for c in ['player_name','name','id','player_id'] if c in X.columns]
X = X.drop(columns=noise_cols, errors='ignore')
print('Dropped:', noise_cols)

Dropped: ['name']


## 5. Handle Missing Values

In [5]:
numeric_cols = X.select_dtypes(include=np.number).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
X.isnull().sum().sort_values(ascending=False).head()

,0
Unnamed: 0,0
gp,0
min,0
pts,0
fgm,0


## 6. Correlation Analysis

In [6]:
corr = X.select_dtypes(include=np.number).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
print('Highly correlated columns:', to_drop)
X_reduced = X.drop(columns=to_drop)

Highly correlated columns: ['pts', 'fgm', 'fga', '3pa', 'fta', 'reb']


## 7. Create Composite Features

In [7]:
if {'pts','min'}.issubset(X_reduced.columns):
    X_reduced['points_per_minute'] = X_reduced['pts'] / (X_reduced['min'] + 1e-6)

if {'pts','reb','ast'}.issubset(X_reduced.columns):
    X_reduced['impact_score'] = X_reduced['pts'] + X_reduced['reb'] + X_reduced['ast']

X_reduced.head()

,Unnamed: 0,gp,min,fg,3p_made,3p,ftm,ft,oreb,dreb,ast,stl,blk,tov
0,0,36,27.4,34.7,0.5,25.0,1.6,69.9,0.7,3.4,1.9,0.4,0.4,1.3
1,1,35,26.9,29.6,0.7,23.5,2.6,76.5,0.5,2.0,3.7,1.1,0.5,1.6
2,2,74,15.3,42.2,0.4,24.4,0.9,67.0,0.5,1.7,1.0,0.5,0.3,1.0
3,3,58,11.6,42.6,0.1,22.6,0.9,68.9,1.0,0.9,0.8,0.6,0.1,1.0
4,4,48,11.5,52.4,0.0,0.0,1.3,67.4,1.0,1.5,0.3,0.3,0.4,0.8


## 8. Final Dataset

In [8]:
final_df = pd.concat([X_reduced, y], axis=1)
final_df.to_csv('nba_feature_engineered.csv', index=False)
print(final_df.shape)
final_df.head()

(1340, 15)


,Unnamed: 0,gp,min,fg,3p_made,3p,ftm,ft,oreb,dreb,ast,stl,blk,tov,target_5yrs
0,0,36,27.4,34.7,0.5,25.0,1.6,69.9,0.7,3.4,1.9,0.4,0.4,1.3,0
1,1,35,26.9,29.6,0.7,23.5,2.6,76.5,0.5,2.0,3.7,1.1,0.5,1.6,0
2,2,74,15.3,42.2,0.4,24.4,0.9,67.0,0.5,1.7,1.0,0.5,0.3,1.0,0
3,3,58,11.6,42.6,0.1,22.6,0.9,68.9,1.0,0.9,0.8,0.6,0.1,1.0,1
4,4,48,11.5,52.4,0.0,0.0,1.3,67.4,1.0,1.5,0.3,0.3,0.4,0.8,1


## 9. Summary
- Target variable isolated.
- Noise columns removed.
- Missing values handled.
- Highly correlated features reduced.
- Composite features engineered.
- Dataset exported for machine learning.